# Venom API / KPI Tutorial

This notebook is an end-to-end guide to using Venom. In this project context, **KPI** means the practical checks that show the package is usable: training starts, checkpoints are saved, sampling writes images, each model family exposes a consistent Python API, and guidance/conditioning paths can be called correctly.

Venom currently covers six families of generative models:

- Diffusion / Score SDE / Flow Matching / One-step generation: `venom.diffusion`
- VAE family: `venom.vae`
- Normalizing flows: `venom.flows`
- GAN family: `venom.gan`
- Energy-based models: `venom.ebm`
- Shared MNIST data and utilities: `venom.data`, `venom.utils`

All examples use MNIST by default. Training outputs are written under `runs/`. Most commands below use `--epochs 1` for quick validation; for real experiments, increase this to `5`, `20`, or more.


## 0. Installation And Environment Check

Run this notebook from the project root or from the `notebooks/` directory. If this is your first time using Venom, install it as an editable package.


In [ ]:
# If Venom is not installed yet, uncomment the next line.
# %pip install -e .

import os
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(PROJECT_ROOT)
print("Project root:", PROJECT_ROOT)
print("Current working directory:", Path.cwd())
print("Python:", sys.version)

In [ ]:
import torch
import venom

print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
print("venom top-level exports loaded:", len(venom.__all__))

## 1. Overall KPI Checklist

Each model family is validated with the same minimal checklist:

| KPI | Expected result |
|---|---|
| Training entrypoint | The corresponding `train_*.py` starts and prints loss or bits-per-dim. |
| Checkpoint | `runs/<family>/<variant>/model_001.pt` is saved. |
| Preview grid | `samples_001.png` is saved. |
| Sampling entrypoint | The corresponding `sample_*.py` loads a checkpoint and saves an image grid. |
| Python API | `build_mnist_*` or model classes can call `training_loss`, `sample`, or `log_prob`. |
| Guidance / conditioning | Diffusion classifier guidance / CFG and label conditioning paths for VAE/GAN/EBM can be called through `--labels` or `y`. |


## 2. Diffusion / Score / Flow Matching / One-step Models

These models live in `venom.diffusion`. Use `train_diffusion.py` for training and `sample_diffusion.py` for sampling.

Core variants:

- Discrete diffusion: `ddpm`, `improved-ddpm`, `adm`, `cfg`
- Continuous score/SDE: `ncsn`, `ncsnv2`, `score-sde-vp`, `score-sde-ve`, `score-sde-subvp`
- EDM/PFGM: `edm`, `pfgm`, `pfgm++`
- Flow matching family: `rectified-flow`, `flow-matching`, `conditional-flow-matching`, `ot-cfm`, `stochastic-interpolants`
- One-step/few-step family: `consistency`, `shortcut`, `meanflow`
- Backbones: `--backbone unet` or `--backbone dit`


In [ ]:
# Fast smoke test: train DDPM for 1 epoch.
!python train_diffusion.py --variant ddpm --epochs 1 --batch-size 128 --sample-steps 20

In [ ]:
# Sample from a DDPM-family checkpoint with DDIM.
!python sample_diffusion.py \
  --checkpoint runs/mnist_diffusion/ddpm/model_001.pt \
  --sampler ddim \
  --sample-steps 20 \
  --num-samples 64 \
  --out runs/mnist_diffusion/ddpm/notebook_ddim_samples.png

### Diffusion Guidance

Venom supports three main diffusion guidance paths:

1. **Class conditioning**: train `adm` directly with class labels.
2. **Classifier-free guidance**: train `cfg` with `--class-dropout`, then sample with `--guidance-scale`.
3. **Classifier guidance**: train a noised classifier first, then sample with `--classifier-checkpoint` and `--classifier-scale`.


In [ ]:
# Train a classifier-free guidance model.
!python train_diffusion.py --variant cfg --epochs 1 --class-dropout 0.1 --sample-steps 20

In [ ]:
# Sample with classifier-free guidance.
!python sample_diffusion.py \
  --checkpoint runs/mnist_diffusion/cfg/model_001.pt \
  --sampler dpm-solver++ \
  --sample-steps 20 \
  --labels 0,1,2,3,4,5,6,7,8,9 \
  --guidance-scale 3.0 \
  --out runs/mnist_diffusion/cfg/notebook_cfg_samples.png

In [ ]:
# Classifier guidance path: train a classifier, then sample an ADM/DDPM-family checkpoint with it.
# These commands are a bit slower, so run them only when needed.
# !python -m venom.diffusion.train_classifier_mnist --epochs 1
# !python sample_diffusion.py \
#   --checkpoint runs/mnist_diffusion/adm/model_001.pt \
#   --sampler ddpm \
#   --labels 0,1,2,3,4,5,6,7,8,9 \
#   --classifier-checkpoint runs/mnist_diffusion/classifier/classifier_001.pt \
#   --classifier-scale 1.0

In [ ]:
# Python API: build and call training_loss / sampler directly.
import torch
from venom.diffusion.factory import build_mnist_diffusion
from venom.diffusion.samplers import make_sampler

model, diffusion = build_mnist_diffusion("ddpm", timesteps=1000, backbone="unet")
images = torch.randn(4, 1, 28, 28)
loss = diffusion.training_loss(images)
sampler = make_sampler(diffusion, "ddim", steps=10)
print("loss:", float(loss.detach()))

## 3. VAE Family

VAE models live in `venom.vae`. Use `train_vae.py` for training and `sample_vae.py` for sampling.

Supported variants: `vae`, `conv-vae`, `beta-vae`, `cvae`, `iwae`, `vq-vae`, `ladder-vae`, `hierarchical-vae`, `flow-vae`.

`cvae` supports label conditioning, so you can pass `--labels` during sampling.


In [ ]:
!python train_vae.py --variant conv-vae --epochs 1 --batch-size 128

In [ ]:
!python sample_vae.py \
  --checkpoint runs/mnist_vae/conv-vae/model_001.pt \
  --num-samples 64 \
  --out runs/mnist_vae/conv-vae/notebook_samples.png

In [ ]:
# Conditional VAE example.
# !python train_vae.py --variant cvae --epochs 1
# !python sample_vae.py \
#   --checkpoint runs/mnist_vae/cvae/model_001.pt \
#   --labels 0,1,2,3,4,5,6,7,8,9

In [ ]:
from venom.vae.factory import build_mnist_vae

vae = build_mnist_vae("conv-vae", latent_dim=64)
images = torch.randn(4, 1, 28, 28)
loss = vae.training_loss(images)
samples = vae.sample(batch_size=4, device=images.device)
print("vae loss:", float(loss.detach()), "samples:", tuple(samples.shape))

## 4. Normalizing Flows

Normalizing flows live in `venom.flows`. They are separate from `venom.diffusion.flow_matching`: these models use invertible transformations and change-of-variables likelihoods.

Supported variants: `planar-flow`, `radial-flow`, `nice`, `realnvp`, `glow`, `maf`, `iaf`, `neural-spline-flow`, `ffjord`, `flow++`.

The core KPI is that bits-per-dim decreases and `sample_flow.py` maps Gaussian latents back into image space.


In [ ]:
!python train_flow.py --variant realnvp --epochs 1 --batch-size 128 --num-layers 4 --hidden-dim 256

In [ ]:
!python sample_flow.py \
  --checkpoint runs/mnist_flow/realnvp/model_001.pt \
  --num-samples 64 \
  --out runs/mnist_flow/realnvp/notebook_samples.png

In [ ]:
from venom.flows import build_mnist_flow

flow, config = build_mnist_flow("realnvp", hidden_dim=256, num_layers=4)
images = torch.randn(4, 1, 28, 28)
bpd = flow.training_loss(images)
samples = flow.sample(batch_size=4, device=images.device)
print(config)
print("bits/dim:", float(bpd.detach()), "samples:", tuple(samples.shape))

## 5. GAN Family

GAN models live in `venom.gan`. Use `train_gan.py` for training and `sample_gan.py` for sampling.

Supported variants: `gan`, `dcgan`, `cgan`, `acgan`, `infogan`, `lsgan`, `wgan`, `wgan-gp`, `hinge-gan`, `sn-gan`.

Conditioning: `cgan` and `acgan` support labels; pass `--labels` during sampling.


In [ ]:
!python train_gan.py --variant dcgan --epochs 1 --batch-size 128

In [ ]:
!python sample_gan.py \
  --checkpoint runs/mnist_gan/dcgan/model_001.pt \
  --num-samples 64 \
  --out runs/mnist_gan/dcgan/notebook_samples.png

In [ ]:
# Conditional GAN example.
# !python train_gan.py --variant cgan --epochs 1
# !python sample_gan.py \
#   --checkpoint runs/mnist_gan/cgan/model_001.pt \
#   --labels 0,1,2,3,4,5,6,7,8,9

In [ ]:
from venom.gan import build_mnist_gan

generator, discriminator, gan_config = build_mnist_gan("dcgan", latent_dim=128)
z = torch.randn(4, 128)
fake = generator(z)
disc_out = discriminator(fake)
print(gan_config)
print("fake:", tuple(fake.shape), "logits:", tuple(disc_out["logits"].shape))

## 6. Energy-Based Models

EBM models live in `venom.ebm`. Use `train_ebm.py` for training and `sample_ebm.py` for sampling.

Supported variants: `rbm`, `gaussian-rbm`, `conditional-rbm`, `conv-rbm`, `deep-ebm`, `conditional-ebm`, `jem`, `score-matching-ebm`, `sliced-score-matching-ebm`, `nce-ebm`.

Sampling: RBM-family models use Gibbs sampling; deep EBM/JEM models use Langevin/SGLD. `conditional-rbm` and `conditional-ebm` support labels.


In [ ]:
!python train_ebm.py --variant rbm --epochs 1 --batch-size 128 --cd-steps 1

In [ ]:
!python sample_ebm.py \
  --checkpoint runs/mnist_ebm/rbm/model_001.pt \
  --steps 100 \
  --num-samples 64 \
  --out runs/mnist_ebm/rbm/notebook_samples.png

In [ ]:
from venom.ebm import RBM, cd_loss

rbm = RBM(hidden_dim=128)
images = torch.rand(4, 1, 28, 28)
loss, negatives = cd_loss(rbm, images, steps=1)
print("cd loss:", float(loss.detach()), "negatives:", tuple(negatives.shape))

## 7. Recommended Smoke-Test Order

If you only want to verify that the whole package works, run these in order:

1. `python train_vae.py --variant conv-vae --epochs 1`
2. `python train_gan.py --variant dcgan --epochs 1`
3. `python train_flow.py --variant realnvp --epochs 1 --num-layers 4 --hidden-dim 256`
4. `python train_ebm.py --variant rbm --epochs 1`
5. `python train_diffusion.py --variant ddpm --epochs 1 --sample-steps 20`
6. `python train_diffusion.py --variant cfg --epochs 1 --class-dropout 0.1 --sample-steps 20`

These six runs cover reconstruction, adversarial training, exact likelihood, energy training, diffusion sampling, and CFG guidance.


## 8. Output Locations

| Model family | Checkpoint directory | Sampling script |
|---|---|---|
| Diffusion / Score / Flow Matching / One-step | `runs/mnist_diffusion/<variant>/` | `sample_diffusion.py` |
| VAE | `runs/mnist_vae/<variant>/` | `sample_vae.py` |
| Normalizing Flow | `runs/mnist_flow/<variant>/` | `sample_flow.py` |
| GAN | `runs/mnist_gan/<variant>/` | `sample_gan.py` |
| EBM | `runs/mnist_ebm/<variant>/` | `sample_ebm.py` |

Each training script usually saves:

- `model_001.pt`, `model_002.pt`, ...
- `samples_001.png`, `samples_002.png`, ...

This is the most basic project KPI: train, save, sample, and reproduce each API path.
